# 🐞 Pest Forecasting with Meteorological Data  
## Information Systems & Business Intelligence Final Project

Welcome to our final project! In this series, we demonstrate how data science and business intelligence can help predict pest outbreaks using weather and capture data.

---

## 👨‍💻 Meet the Team

**Seyyedalireza Khosh Solat**  
- Designed ML model tournaments architecture  
- Implemented advanced ML algorithms  
- Developed time series forecasting components  
- Led model evaluation methodology

**Giovanni Montanile**  
- Developed Power BI dashboards  
- Created business intelligence solutions  
- Designed interactive visualizations  
- Implemented data reporting systems

**Farshad Farahtaj**  
- Led data preprocessing workflow  
- Conducted exploratory data analysis  
- Performed feature engineering  
- Implemented data validation protocols

---

## 📖 Project Overview

Our goal is to analyze meteorological and pest capture data to build predictive models for:
- **Regression:** Predicting the exact number of insects caught each day.
- **Classification:** Predicting whether there will be new pest captures (yes/no).

We use a combination of data science and business intelligence techniques to turn raw data into actionable insights for pest management.

---

## 🚦 Notebook 1: Data Preprocessing & Exploratory Data Analysis (EDA)

**What you'll learn in this notebook:**
1. **How we load and check our raw datasets**
2. **How we clean, validate, and prepare the data for analysis**
3. **How we explore the data visually and statistically**
4. **How we create clean datasets ready for machine learning and forecasting**

We explain each step in simple terms, so you can follow along even if you're new to data science!

---

In [2]:
# Import Essential Libraries for Data Science
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
from plotly.subplots import make_subplots
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"Plotly version: Interactive visualizations enabled!")

✓ Libraries imported successfully!
Pandas version: 2.2.2
NumPy version: 1.26.4
Plotly version: Interactive visualizations enabled!


## 1. Data Loading

Before we can analyze or model anything, we need to load our data!

We use two main datasets:
- **complete_merged_data.csv**: Contains daily weather and pest capture data for all locations (used for time-series analysis).
- **complete_engineered_pest_data.csv**: Contains extra features we created for machine learning models.

Let's load both and take a first look at what they contain.

## 📂 Why Two Datasets?

We use two different datasets in this project:

- **Merged Data:** Contains the raw daily records for each location, including weather and pest captures. This is best for time-series analysis and understanding trends over time.
- **Engineered Data:** Contains extra features we created (like monthly or seasonal indicators) to help machine learning models learn better patterns.

Using both lets us compare traditional time-series forecasting with modern machine learning approaches.

---

## 📝 What is Feature Engineering?

Feature engineering means creating new columns (features) from the original data to help our models learn better.

For example, we might:
- Calculate the temperature range for each day
- Add columns for "Month" or "Season"
- Create binary columns like "New catches" (1 if any insects were caught, 0 otherwise)

These new features can make a big difference in model performance!

---

In [3]:
# Load the datasets
try:
    # Time-series dataset for SARIMAX modeling
    merged_data = pd.read_csv('complete_merged_data.csv')
    
    # Feature-engineered dataset for ML modeling
    engineered_data = pd.read_csv('complete_engineered_pest_data.csv')
    
    print("✓ Datasets loaded successfully!")
    print(f"\nMerged dataset shape: {merged_data.shape}")
    print(f"Engineered dataset shape: {engineered_data.shape}")
    
except FileNotFoundError as e:
    print(f"❌ Error loading files: {e}")
    print("Please ensure the CSV files are in the current directory.")

✓ Datasets loaded successfully!

Merged dataset shape: (245, 9)
Engineered dataset shape: (245, 17)


In [4]:
# Examine the structure of both datasets
print("=" * 60)
print("MERGED DATASET (Time-Series Data)")
print("=" * 60)
print("\nFirst 5 rows:")
print(merged_data.head())
print("\nDataset Info:")
print(merged_data.info())
print("\nColumn names:")
print(list(merged_data.columns))

MERGED DATASET (Time-Series Data)

First 5 rows:
         Date    Location  Average Temperature  Temperature Range (Low)  \
0  06.07.2024  Cicalino 1                22.34                    21.53   
1  06.07.2024  Cicalino 2                26.17                    25.28   
2  06.07.2024     Imola 1                29.68                    29.11   
3  06.07.2024     Imola 2                28.83                    28.04   
4  06.07.2024     Imola 3                26.89                    25.77   

   Temperature Range (High)  Average Humidity  Number of insects  New catches  \
0                     23.12             72.25                0.0          0.0   
1                     27.00             56.06                0.0          0.0   
2                     30.24             42.93                0.0          0.0   
3                     29.76             52.45                0.0          0.0   
4                     27.85             64.88                0.0          0.0   

   Event  
0 

In [5]:
print("=" * 60)
print("ENGINEERED DATASET (Feature-Rich Data)")
print("=" * 60)
print("\nFirst 5 rows:")
print(engineered_data.head())
print("\nDataset Info:")
print(engineered_data.info())
print("\nColumn names:")
print(list(engineered_data.columns))

ENGINEERED DATASET (Feature-Rich Data)

First 5 rows:
         Date    Location  Location_Code  Average Temperature  \
0  2024-07-06  Cicalino 1              0                22.34   
1  2024-07-07  Cicalino 1              0                23.52   
2  2024-07-08  Cicalino 1              0                25.67   
3  2024-07-09  Cicalino 1              0                25.87   
4  2024-07-10  Cicalino 1              0                26.41   

   Average Humidity  Temp_Range  Temp_Avg_3d  Humidity_Avg_3d  Insects_Lag1  \
0             72.25        1.59    22.340000        72.250000           0.0   
1             76.73        1.22    22.930000        74.490000           0.0   
2             69.14        1.87    23.843333        72.706667           0.0   
3             53.65        1.98    25.020000        66.506667           0.0   
4             58.94        1.91    25.983333        60.576667           0.0   

   Insects_Lag3  Recent_Activity  Days_Since_Cleaning  Month  Day      Season  \

## 2. Data Cleaning & Preprocessing

Raw data is often messy!  
To make sure our analysis and models are reliable, we need to:
1. **Fix date formats** so we can analyze trends over time.
2. **Find and handle missing values** (empty cells).
3. **Check data types** (numbers, text, dates).
4. **Remove duplicates** so every row is unique.

We'll walk through these steps for both datasets, explaining what we do and why.

In [6]:
# Clean the merged dataset (for time-series analysis)
print("Cleaning MERGED DATASET...")
print("=" * 40)

# Convert Date column to datetime
merged_data['Date'] = pd.to_datetime(merged_data['Date'], format='%d.%m.%Y', errors='coerce')
print(f"✓ Date column converted to datetime")

# Set Date as index for time-series operations
merged_data_clean = merged_data.copy()
merged_data_clean.set_index('Date', inplace=True)
print(f"✓ Date set as index")

# Check for missing values
missing_values = merged_data_clean.isnull().sum()
print(f"\nMissing values per column:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values found!")

# Check for duplicates
duplicates = merged_data_clean.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

print(f"\nCleaned merged dataset shape: {merged_data_clean.shape}")

Cleaning MERGED DATASET...
✓ Date column converted to datetime
✓ Date set as index

Missing values per column:
No missing values found!

Duplicate rows: 96

Cleaned merged dataset shape: (245, 8)


In [7]:
# Clean the engineered dataset (for ML modeling)
print("\nCleaning ENGINEERED DATASET...")
print("=" * 40)

# Convert Date column to datetime
engineered_data['Date'] = pd.to_datetime(engineered_data['Date'], errors='coerce')
print(f"✓ Date column converted to datetime")

# Create a clean copy
engineered_data_clean = engineered_data.copy()

# Check for missing values
missing_values = engineered_data_clean.isnull().sum()
print(f"\nMissing values per column:")
print(missing_values[missing_values > 0] if missing_values.sum() > 0 else "No missing values found!")

# Check for duplicates
duplicates = engineered_data_clean.duplicated().sum()
print(f"\nDuplicate rows: {duplicates}")

# Check data types
print(f"\nData types:")
print(engineered_data_clean.dtypes)

print(f"\nCleaned engineered dataset shape: {engineered_data_clean.shape}")


Cleaning ENGINEERED DATASET...
✓ Date column converted to datetime

Missing values per column:
No missing values found!

Duplicate rows: 0

Data types:
Date                   datetime64[ns]
Location                       object
Location_Code                   int64
Average Temperature           float64
Average Humidity              float64
Temp_Range                    float64
Temp_Avg_3d                   float64
Humidity_Avg_3d               float64
Insects_Lag1                  float64
Insects_Lag3                  float64
Recent_Activity                 int64
Days_Since_Cleaning             int64
Month                           int64
Day                             int64
Season                         object
Number of insects             float64
New catches                   float64
dtype: object

Cleaned engineered dataset shape: (245, 17)


## 3. Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) helps us understand our data before modeling.

In this section, we will:
- Visualize how many insects are caught and when
- See how pest activity changes over time and by location
- Explore how weather and pest captures are related

Clear visualizations and simple statistics will help us spot patterns and prepare for modeling.

In [8]:
# 3.1 Target Variable Analysis
print("TARGET VARIABLE ANALYSIS")
print("=" * 50)

# Create interactive subplots for target variable analysis
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Distribution of Insect Counts', 'Insect Counts by Location', 
                   'Distribution of New Catches (Binary Target)', 'Daily Insect Activity Over Time'),
    specs=[[{"type": "histogram"}, {"type": "box"}],
           [{"type": "pie"}, {"type": "scatter"}]]
)

# 1. Interactive histogram of insect counts
fig_hist = px.histogram(engineered_data_clean, x='Number of insects', 
                       title='Distribution of Insect Counts',
                       labels={'Number of insects': 'Number of Insects', 'count': 'Frequency'},
                       nbins=30)
for trace in fig_hist.data:
    fig.add_trace(trace, row=1, col=1)

# 2. Interactive box plot by location
fig_box = px.box(engineered_data_clean, x='Location', y='Number of insects',
                title='Insect Counts by Location',
                labels={'Number of insects': 'Number of Insects'})
for trace in fig_box.data:
    fig.add_trace(trace, row=1, col=2)

# 3. Interactive pie chart for binary catches
binary_catches = (engineered_data_clean['New catches'] > 0).astype(int).value_counts()
fig.add_trace(go.Pie(labels=['No New Catches', 'New Catches'], 
                    values=binary_catches.values,
                    name="New Catches Distribution"), row=2, col=1)

# 4. Interactive time series plot
daily_stats = engineered_data_clean.groupby('Date')['Number of insects'].agg(['sum', 'mean', 'max']).reset_index()
fig.add_trace(go.Scatter(x=daily_stats['Date'], y=daily_stats['sum'],
                        mode='lines', name='Total Daily',
                        hovertemplate='Date: %{x}<br>Total Insects: %{y}<extra></extra>'), row=2, col=2)
fig.add_trace(go.Scatter(x=daily_stats['Date'], y=daily_stats['mean'],
                        mode='lines', name='Average Daily',
                        hovertemplate='Date: %{x}<br>Average Insects: %{y:.1f}<extra></extra>'), row=2, col=2)

# Update layout
fig.update_layout(height=800, showlegend=True, 
                 title_text="Target Variable Analysis: Number of Insects Caught",
                 title_x=0.5)
fig.show()

# Create separate interactive plots for better visibility
print("\n📊 INTERACTIVE VISUALIZATIONS:")

# Enhanced histogram with hover information
fig_hist_detailed = px.histogram(engineered_data_clean, x='Number of insects', 
                                title='Interactive Distribution of Insect Counts',
                                labels={'Number of insects': 'Number of Insects', 'count': 'Frequency'},
                                nbins=30, color_discrete_sequence=['skyblue'])
fig_hist_detailed.update_traces(hovertemplate='Insects: %{x}<br>Frequency: %{y}<extra></extra>')
fig_hist_detailed.update_layout(height=400)
fig_hist_detailed.show()

# Enhanced box plot with detailed hover
fig_box_detailed = px.box(engineered_data_clean, x='Location', y='Number of insects',
                         title='Interactive Insect Counts by Location',
                         labels={'Number of insects': 'Number of Insects'},
                         color='Location')
fig_box_detailed.update_traces(hovertemplate='Location: %{x}<br>Insects: %{y}<br>%{text}<extra></extra>')
fig_box_detailed.update_layout(height=400)
fig_box_detailed.show()

# Interactive time series with enhanced features
fig_time = px.line(daily_stats, x='Date', y=['sum', 'mean'], 
                  title='Interactive Daily Insect Activity Over Time',
                  labels={'value': 'Number of Insects', 'variable': 'Metric'})
fig_time.update_traces(hovertemplate='Date: %{x}<br>Value: %{y:.1f}<extra></extra>')
fig_time.update_layout(height=400)
fig_time.show()

# Print summary statistics
print(f"\nSUMMARY STATISTICS for 'Number of insects':")
print(engineered_data_clean['Number of insects'].describe())
print(f"\nZero captures: {(engineered_data_clean['Number of insects'] == 0).sum()} out of {len(engineered_data_clean)} records ({(engineered_data_clean['Number of insects'] == 0).mean()*100:.1f}%)")
print(f"Binary target distribution: {binary_catches.values[1]:.0f} days with catches, {binary_catches.values[0]:.0f} days without")

TARGET VARIABLE ANALYSIS



📊 INTERACTIVE VISUALIZATIONS:



SUMMARY STATISTICS for 'Number of insects':
count    245.000000
mean       0.244898
std        0.693371
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        5.000000
Name: Number of insects, dtype: float64

Zero captures: 207 out of 245 records (84.5%)
Binary target distribution: 21 days with catches, 224 days without


In [9]:
# 3.2 Time Series Analysis
print("\n" + "=" * 50)
print("TIME SERIES ANALYSIS")
print("=" * 50)

# Prepare data for time series visualization
daily_totals = merged_data_clean.groupby('Date')['Number of insects'].sum().reset_index()
daily_totals['MA_7'] = daily_totals['Number of insects'].rolling(window=7, center=True).mean()

# 1. Interactive total daily insect captures
fig_total = px.line(daily_totals, x='Date', y='Number of insects',
                   title='Interactive Total Daily Insect Captures (All Locations)',
                   labels={'Number of insects': 'Total Insects Caught', 'Date': 'Date'})
fig_total.update_traces(hovertemplate='Date: %{x}<br>Total Insects: %{y}<extra></extra>',
                       line=dict(color='darkblue', width=2))
fig_total.update_layout(height=400, showlegend=False)
fig_total.show()

# 2. Interactive captures by location
print("\n📊 By Location Analysis:")
location_data_list = []
for location in merged_data_clean['Location'].unique():
    location_data = merged_data_clean[merged_data_clean['Location'] == location]
    daily_location = location_data.groupby('Date')['Number of insects'].sum().reset_index()
    daily_location['Location'] = location
    location_data_list.append(daily_location)

all_location_data = pd.concat(location_data_list, ignore_index=True)

fig_locations = px.line(all_location_data, x='Date', y='Number of insects', color='Location',
                       title='Interactive Daily Insect Captures by Location',
                       labels={'Number of insects': 'Insects Caught', 'Date': 'Date'})
fig_locations.update_traces(hovertemplate='Date: %{x}<br>Location: %{fullData.name}<br>Insects: %{y}<extra></extra>')
fig_locations.update_layout(height=500)
fig_locations.show()

# 3. Interactive trend analysis with moving average
print("\n📈 Trend Analysis:")
fig_trend = go.Figure()

# Add daily totals
fig_trend.add_trace(go.Scatter(x=daily_totals['Date'], y=daily_totals['Number of insects'],
                              mode='lines', name='Daily Total', opacity=0.5,
                              line=dict(color='lightblue'),
                              hovertemplate='Date: %{x}<br>Daily Total: %{y}<extra></extra>'))

# Add moving average
fig_trend.add_trace(go.Scatter(x=daily_totals['Date'], y=daily_totals['MA_7'],
                              mode='lines', name='7-Day Moving Average',
                              line=dict(color='red', width=3),
                              hovertemplate='Date: %{x}<br>7-Day Average: %{y:.1f}<extra></extra>'))

fig_trend.update_layout(title='Interactive Trend Analysis with 7-Day Moving Average',
                       xaxis_title='Date',
                       yaxis_title='Insects Caught',
                       height=500)
fig_trend.show()

# 4. Interactive peak activity heatmap
print("\n🔥 Peak Activity Calendar View:")
# Create calendar heatmap data
daily_totals['Year'] = daily_totals['Date'].dt.year
daily_totals['Month'] = daily_totals['Date'].dt.month
daily_totals['Day'] = daily_totals['Date'].dt.day

# Create a pivot table for heatmap
calendar_data = daily_totals.pivot_table(values='Number of insects', 
                                        index='Day', 
                                        columns='Month', 
                                        aggfunc='mean')

fig_heatmap = px.imshow(calendar_data, 
                       title='Interactive Calendar Heatmap: Average Daily Pest Activity',
                       labels=dict(x="Month", y="Day", color="Avg Insects"),
                       aspect="auto",
                       color_continuous_scale="Viridis")
fig_heatmap.update_traces(hovertemplate='Month: %{x}<br>Day: %{y}<br>Avg Insects: %{z:.1f}<extra></extra>')
fig_heatmap.update_layout(height=400)
fig_heatmap.show()

# Print peak activity periods
print(f"\nPEAK ACTIVITY ANALYSIS:")
top_days = daily_totals.nlargest(5, 'Number of insects')[['Date', 'Number of insects']]
print("Top 5 days with highest pest activity:")
for idx, row in top_days.iterrows():
    print(f"  {row['Date'].strftime('%Y-%m-%d')}: {row['Number of insects']:.0f} insects")

# Activity statistics by month
monthly_stats = daily_totals.groupby('Month')['Number of insects'].agg(['mean', 'max', 'sum']).round(2)
print(f"\nMonthly Activity Summary:")
print(monthly_stats)


TIME SERIES ANALYSIS

TIME SERIES ANALYSIS



📊 By Location Analysis:



📈 Trend Analysis:



🔥 Peak Activity Calendar View:



PEAK ACTIVITY ANALYSIS:
Top 5 days with highest pest activity:
  2024-08-23: 9 insects
  2024-08-22: 6 insects
  2024-08-20: 5 insects
  2024-07-22: 4 insects
  2024-07-20: 3 insects

Monthly Activity Summary:
       mean  max   sum
Month                 
7      1.15  4.0  30.0
8      1.30  9.0  30.0


In [10]:
# 3.3 Weather-Pest Correlation Analysis
print("\n" + "=" * 50)
print("WEATHER-PEST CORRELATION ANALYSIS")
print("=" * 50)

# Select numeric columns for correlation analysis
numeric_cols = engineered_data_clean.select_dtypes(include=[np.number]).columns
corr_matrix = engineered_data_clean[numeric_cols].corr()

# Create interactive correlation heatmap
print("\n📊 INTERACTIVE CORRELATION MATRIX:")
fig_corr_full = px.imshow(corr_matrix, 
                         title='Interactive Full Feature Correlation Matrix',
                         labels=dict(color="Correlation"),
                         aspect="auto",
                         color_continuous_scale="RdBu_r",
                         zmin=-1, zmax=1)
fig_corr_full.update_traces(hovertemplate='Feature 1: %{x}<br>Feature 2: %{y}<br>Correlation: %{z:.3f}<extra></extra>')
fig_corr_full.update_layout(height=600)
fig_corr_full.show()

# Focus on correlations with target variables
print("\n🎯 TARGET CORRELATIONS:")
target_corr = corr_matrix[['Number of insects', 'New catches']].sort_values('Number of insects', key=abs, ascending=False)

fig_target_corr = px.imshow(target_corr.T, 
                           title='Interactive Correlations with Target Variables',
                           labels=dict(x="Features", y="Target Variables", color="Correlation"),
                           aspect="auto",
                           color_continuous_scale="RdBu_r",
                           zmin=-1, zmax=1)
fig_target_corr.update_traces(hovertemplate='Feature: %{x}<br>Target: %{y}<br>Correlation: %{z:.3f}<extra></extra>')
fig_target_corr.update_layout(height=300, xaxis={'tickangle': 45})
fig_target_corr.show()

# Interactive correlation bar chart for top correlations
print("\n🔍 TOP CORRELATIONS:")
# Get top correlations with insect counts
insect_corr = corr_matrix['Number of insects'].drop('Number of insects').sort_values(key=abs, ascending=False)
top_correlations = insect_corr.head(10)

fig_bar_corr = px.bar(x=top_correlations.values, y=top_correlations.index, 
                     orientation='h',
                     title='Top 10 Feature Correlations with Number of Insects',
                     labels={'x': 'Correlation Coefficient', 'y': 'Features'},
                     color=top_correlations.values,
                     color_continuous_scale="RdBu_r")
fig_bar_corr.update_traces(hovertemplate='Feature: %{y}<br>Correlation: %{x:.3f}<extra></extra>')
fig_bar_corr.update_layout(height=500)
fig_bar_corr.show()

# Interactive scatter plot matrix for key relationships
print("\n🔄 KEY RELATIONSHIPS:")
key_features = ['Number of insects', 'Average Temperature', 'Average Humidity']
if len(key_features) > 1 and all(feat in engineered_data_clean.columns for feat in key_features):
    fig_scatter_matrix = px.scatter_matrix(engineered_data_clean[key_features],
                                          title='Interactive Scatter Matrix: Key Weather-Pest Relationships',
                                          color='Number of insects',
                                          color_continuous_scale="Viridis")
    fig_scatter_matrix.update_traces(diagonal_visible=False)
    fig_scatter_matrix.update_layout(height=600)
    fig_scatter_matrix.show()

# Print strongest correlations with insect counts
print("\nSTRONGEST CORRELATIONS with 'Number of insects':")
insect_corr = corr_matrix['Number of insects'].sort_values(key=abs, ascending=False)
for feature, corr in insect_corr.items():
    if feature != 'Number of insects' and abs(corr) > 0.1:
        direction = "UP" if corr > 0 else "DOWN"
        strength = "Strong" if abs(corr) > 0.5 else "Moderate" if abs(corr) > 0.3 else "Weak"
        print(f"  {direction} {feature}: {corr:.3f} ({strength})")

print("\nSTRONGEST CORRELATIONS with 'New catches':")
catch_corr = corr_matrix['New catches'].sort_values(key=abs, ascending=False)
for feature, corr in catch_corr.items():
    if feature != 'New catches' and abs(corr) > 0.1:
        direction = "UP" if corr > 0 else "DOWN"
        strength = "Strong" if abs(corr) > 0.5 else "Moderate" if abs(corr) > 0.3 else "Weak"
        print(f"  {direction} {feature}: {corr:.3f} ({strength})")


WEATHER-PEST CORRELATION ANALYSIS

📊 INTERACTIVE CORRELATION MATRIX:



🎯 TARGET CORRELATIONS:



🔍 TOP CORRELATIONS:



🔄 KEY RELATIONSHIPS:



STRONGEST CORRELATIONS with 'Number of insects':
  UP New catches: 0.761 (Strong)
  UP Insects_Lag1: 0.710 (Strong)
  UP Recent_Activity: 0.708 (Strong)
  UP Insects_Lag3: 0.524 (Strong)
  UP Average Humidity: 0.332 (Moderate)
  UP Humidity_Avg_3d: 0.321 (Moderate)
  DOWN Temp_Avg_3d: -0.292 (Weak)
  DOWN Average Temperature: -0.282 (Weak)
  UP Day: 0.236 (Weak)
  UP Days_Since_Cleaning: 0.207 (Weak)
  DOWN Temp_Range: -0.182 (Weak)
  DOWN Location_Code: -0.171 (Weak)

STRONGEST CORRELATIONS with 'New catches':
  UP Number of insects: 0.761 (Strong)
  UP Recent_Activity: 0.530 (Strong)
  UP Insects_Lag3: 0.308 (Moderate)
  UP Average Humidity: 0.285 (Weak)
  UP Insects_Lag1: 0.285 (Weak)
  UP Humidity_Avg_3d: 0.258 (Weak)
  DOWN Temp_Avg_3d: -0.258 (Weak)
  DOWN Average Temperature: -0.257 (Weak)
  UP Days_Since_Cleaning: 0.175 (Weak)
  UP Day: 0.174 (Weak)
  DOWN Temp_Range: -0.120 (Weak)
  DOWN Location_Code: -0.115 (Weak)


In [11]:
# 3.4 Weather Patterns and Pest Activity
print("\n" + "=" * 50)
print("WEATHER PATTERNS ANALYSIS")
print("=" * 50)

# Create interactive weather vs pest activity plots
print("\n🌡️ INTERACTIVE WEATHER ANALYSIS:")

# 1. Interactive Temperature vs Insect Count scatter plot
fig_temp = px.scatter(engineered_data_clean, x='Average Temperature', y='Number of insects',
                     title='Interactive Temperature vs Insect Activity',
                     labels={'Average Temperature': 'Average Temperature (°C)', 
                            'Number of insects': 'Number of Insects'},
                     hover_data=['Date', 'Location'],
                     color='Number of insects',
                     color_continuous_scale='Viridis')
fig_temp.update_traces(hovertemplate='Temperature: %{x}°C<br>Insects: %{y}<br>Date: %{customdata[0]}<br>Location: %{customdata[1]}<extra></extra>')
fig_temp.update_layout(height=400)
fig_temp.show()

# 2. Interactive Humidity vs Insect Count scatter plot
fig_humidity = px.scatter(engineered_data_clean, x='Average Humidity', y='Number of insects',
                         title='Interactive Humidity vs Insect Activity', 
                         labels={'Average Humidity': 'Average Humidity (%)', 
                                'Number of insects': 'Number of Insects'},
                         hover_data=['Date', 'Location'],
                         color='Number of insects',
                         color_continuous_scale='Plasma')
fig_humidity.update_traces(hovertemplate='Humidity: %{x}%<br>Insects: %{y}<br>Date: %{customdata[0]}<br>Location: %{customdata[1]}<extra></extra>')
fig_humidity.update_layout(height=400)
fig_humidity.show()

# 3. Interactive Temperature distribution by season (if available)
if 'Season' in engineered_data_clean.columns:
    print("\n🌿 SEASONAL ANALYSIS:")
    fig_season_temp = px.box(engineered_data_clean, x='Season', y='Average Temperature',
                            title='Interactive Temperature Distribution by Season',
                            labels={'Average Temperature': 'Temperature (°C)'},
                            color='Season')
    fig_season_temp.update_traces(hovertemplate='Season: %{x}<br>Temperature: %{y}°C<extra></extra>')
    fig_season_temp.update_layout(height=400)
    fig_season_temp.show()
    
    # Season vs pest activity
    seasonal_activity = engineered_data_clean.groupby('Season')['Number of insects'].agg(['mean', 'sum', 'std']).reset_index()
    fig_season_activity = px.bar(seasonal_activity, x='Season', y='mean',
                                title='Interactive Average Pest Activity by Season',
                                labels={'mean': 'Average Insects per Day'},
                                color='Season')
    fig_season_activity.update_traces(hovertemplate='Season: %{x}<br>Avg Insects: %{y:.1f}<extra></extra>')
    fig_season_activity.update_layout(height=400)
    fig_season_activity.show()
else:
    print("\n📋 Season data not available for analysis")

# 4. Interactive Monthly pest activity (if available)
if 'Month' in engineered_data_clean.columns:
    print("\n📅 MONTHLY ANALYSIS:")
    monthly_activity = engineered_data_clean.groupby('Month')['Number of insects'].agg(['mean', 'sum', 'count']).reset_index()
    
    fig_monthly = px.bar(monthly_activity, x='Month', y='mean',
                        title='Interactive Average Pest Activity by Month',
                        labels={'mean': 'Average Insects per Day', 'Month': 'Month'},
                        color='mean',
                        color_continuous_scale='Viridis')
    fig_monthly.update_traces(hovertemplate='Month: %{x}<br>Avg Insects: %{y:.1f}<extra></extra>')
    fig_monthly.update_layout(height=400)
    fig_monthly.show()
else:
    print("\n📋 Month data not available for analysis")

# 5. Interactive 3D scatter plot: Temperature vs Humidity vs Insects
print("\n🌍 3D WEATHER-PEST RELATIONSHIP:")
fig_3d = px.scatter_3d(engineered_data_clean, x='Average Temperature', y='Average Humidity', z='Number of insects',
                      title='Interactive 3D: Temperature vs Humidity vs Pest Activity',
                      labels={'Average Temperature': 'Temperature (°C)',
                             'Average Humidity': 'Humidity (%)',
                             'Number of insects': 'Number of Insects'},
                      color='Number of insects',
                      color_continuous_scale='Viridis',
                      hover_data=['Date', 'Location'])
fig_3d.update_traces(hovertemplate='Temp: %{x}°C<br>Humidity: %{y}%<br>Insects: %{z}<br>Date: %{customdata[0]}<br>Location: %{customdata[1]}<extra></extra>')
fig_3d.update_layout(height=600)
fig_3d.show()

# Weather summary statistics
print("\nWEATHER SUMMARY STATISTICS:")
weather_stats = engineered_data_clean[['Average Temperature', 'Average Humidity']].describe()
print(weather_stats)

# Correlation summary
temp_corr = engineered_data_clean['Average Temperature'].corr(engineered_data_clean['Number of insects'])
humidity_corr = engineered_data_clean['Average Humidity'].corr(engineered_data_clean['Number of insects'])
print(f"\nWeather-Pest Correlations:")
print(f"Temperature vs Insects: {temp_corr:.3f}")
print(f"Humidity vs Insects: {humidity_corr:.3f}")


WEATHER PATTERNS ANALYSIS

🌡️ INTERACTIVE WEATHER ANALYSIS:



🌿 SEASONAL ANALYSIS:



📅 MONTHLY ANALYSIS:



🌍 3D WEATHER-PEST RELATIONSHIP:



WEATHER SUMMARY STATISTICS:
       Average Temperature  Average Humidity
count           245.000000        245.000000
mean             27.428408         60.354082
std               1.949089         10.458112
min              19.610000         42.930000
25%              26.860000         52.450000
50%              27.250000         62.720000
75%              28.830000         64.880000
max              31.080000         92.100000

Weather-Pest Correlations:
Temperature vs Insects: -0.282
Humidity vs Insects: 0.332


In [12]:
# 3.5 Location-Based Analysis
print("\n" + "=" * 50)
print("LOCATION-BASED ANALYSIS")
print("=" * 50)

# Summary statistics by location
location_summary = engineered_data_clean.groupby('Location').agg({
    'Number of insects': ['count', 'sum', 'mean', 'std', 'max'],
    'New catches': ['sum', 'mean'],
    'Average Temperature': 'mean',
    'Average Humidity': 'mean'
}).round(2)

print("SUMMARY STATISTICS BY LOCATION:")
print(location_summary)

# Interactive location comparison visualizations
print("\n📍 INTERACTIVE LOCATION ANALYSIS:")

# 1. Interactive total insects caught per location
location_totals = engineered_data_clean.groupby('Location')['Number of insects'].sum().sort_values(ascending=False).reset_index()
fig_totals = px.bar(location_totals, x='Location', y='Number of insects',
                   title='Interactive Total Insects Caught by Location',
                   labels={'Number of insects': 'Total Insects', 'Location': 'Location'},
                   color='Number of insects',
                   color_continuous_scale='Blues')
fig_totals.update_traces(hovertemplate='Location: %{x}<br>Total Insects: %{y}<extra></extra>')
fig_totals.update_layout(height=400, xaxis={'tickangle': 45})
fig_totals.show()

# 2. Interactive average daily activity per location
location_avg = engineered_data_clean.groupby('Location')['Number of insects'].mean().sort_values(ascending=False).reset_index()
fig_avg = px.bar(location_avg, x='Location', y='Number of insects',
                title='Interactive Average Daily Activity by Location',
                labels={'Number of insects': 'Average Insects per Day', 'Location': 'Location'},
                color='Number of insects',
                color_continuous_scale='Greens')
fig_avg.update_traces(hovertemplate='Location: %{x}<br>Avg Daily: %{y:.1f}<extra></extra>')
fig_avg.update_layout(height=400, xaxis={'tickangle': 45})
fig_avg.show()

# 3. Interactive success rate per location
success_rate = engineered_data_clean.groupby('Location')['New catches'].mean().sort_values(ascending=False).reset_index()
fig_success = px.bar(success_rate, x='Location', y='New catches',
                    title='Interactive Success Rate (Proportion of Days with Catches)',
                    labels={'New catches': 'Success Rate', 'Location': 'Location'},
                    color='New catches',
                    color_continuous_scale='Reds')
fig_success.update_traces(hovertemplate='Location: %{x}<br>Success Rate: %{y:.3f}<extra></extra>')
fig_success.update_layout(height=400, xaxis={'tickangle': 45}, yaxis=dict(range=[0, 1]))
fig_success.show()

# 4. Interactive weather conditions by location
print("\n🌤️ WEATHER CONDITIONS BY LOCATION:")
location_weather = engineered_data_clean.groupby('Location')[['Average Temperature', 'Average Humidity']].mean().reset_index()

# Temperature and humidity scatter plot
fig_weather_scatter = px.scatter(location_weather, x='Average Temperature', y='Average Humidity',
                               text='Location', size_max=60,
                               title='Interactive Weather Conditions by Location',
                               labels={'Average Temperature': 'Temperature (°C)',
                                      'Average Humidity': 'Humidity (%)'},
                               color='Location')
fig_weather_scatter.update_traces(textposition="top center",
                                hovertemplate='Location: %{text}<br>Temperature: %{x:.1f}°C<br>Humidity: %{y:.1f}%<extra></extra>')
fig_weather_scatter.update_layout(height=500)
fig_weather_scatter.show()

# 5. Interactive location activity over time
print("\n📈 ACTIVITY TRENDS BY LOCATION:")
fig_location_time = px.line(engineered_data_clean, x='Date', y='Number of insects', color='Location',
                           title='Interactive Pest Activity Trends by Location Over Time',
                           labels={'Number of insects': 'Number of Insects', 'Date': 'Date'})
fig_location_time.update_traces(hovertemplate='Date: %{x}<br>Location: %{fullData.name}<br>Insects: %{y}<extra></extra>')
fig_location_time.update_layout(height=500)
fig_location_time.show()

# Summary insights
print("\n🏆 LOCATION PERFORMANCE SUMMARY:")
best_total = location_totals.iloc[0]
best_avg = location_avg.iloc[0]
best_success = success_rate.iloc[0]

print(f"Highest Total Captures: {best_total['Location']} ({best_total['Number of insects']:.0f} insects)")
print(f"Highest Average Daily: {best_avg['Location']} ({best_avg['Number of insects']:.1f} insects/day)")
print(f"Highest Success Rate: {best_success['Location']} ({best_success['New catches']:.1%} of days)")

# Location diversity analysis
total_locations = len(engineered_data_clean['Location'].unique())
total_days = len(engineered_data_clean['Date'].unique())
print(f"\n📊 Dataset Coverage: {total_locations} locations monitored over {total_days} days")


LOCATION-BASED ANALYSIS
SUMMARY STATISTICS BY LOCATION:
           Number of insects                        New catches        \
                       count   sum  mean   std  max         sum  mean   
Location                                                                
Cicalino 1                49  21.0  0.43  0.68  3.0         9.0  0.18   
Cicalino 2                49  14.0  0.29  0.61  2.0         5.0  0.10   
Imola 1                   49  17.0  0.35  1.09  5.0         8.0  0.16   
Imola 2                   49   1.0  0.02  0.14  1.0         1.0  0.02   
Imola 3                   49   7.0  0.14  0.54  3.0         3.0  0.06   

           Average Temperature Average Humidity  
                          mean             mean  
Location                                         
Cicalino 1               26.82            62.84  
Cicalino 2               27.17            61.17  
Imola 1                  28.52            53.10  
Imola 2                  28.05            58.23  
Imola 3 


🌤️ WEATHER CONDITIONS BY LOCATION:



📈 ACTIVITY TRENDS BY LOCATION:



🏆 LOCATION PERFORMANCE SUMMARY:
Highest Total Captures: Cicalino 1 (21 insects)
Highest Average Daily: Cicalino 1 (0.4 insects/day)
Highest Success Rate: Cicalino 1 (18.4% of days)

📊 Dataset Coverage: 5 locations monitored over 49 days


In [13]:
# 3.6 Final Data Validation and Summary
print("\n" + "=" * 60)
print("FINAL DATA VALIDATION & SUMMARY")
print("=" * 60)

# Validate data quality
print("DATA QUALITY CHECKS:")
print(f"✓ Merged dataset: {merged_data_clean.shape[0]} records, {merged_data_clean.shape[1]} features")
print(f"✓ Engineered dataset: {engineered_data_clean.shape[0]} records, {engineered_data_clean.shape[1]} features")
print(f"✓ Date range: {engineered_data_clean['Date'].min().strftime('%Y-%m-%d')} to {engineered_data_clean['Date'].max().strftime('%Y-%m-%d')}")
print(f"✓ Number of locations: {engineered_data_clean['Location'].nunique()}")
print(f"✓ Locations: {list(engineered_data_clean['Location'].unique())}")

# Check target variable distributions
print(f"\nTARGET VARIABLE SUMMARY:")
print(f"• Total insect captures: {engineered_data_clean['Number of insects'].sum():.0f}")
print(f"• Days with captures: {(engineered_data_clean['Number of insects'] > 0).sum()} out of {len(engineered_data_clean)} ({(engineered_data_clean['Number of insects'] > 0).mean()*100:.1f}%)")
print(f"• Days with new catches: {engineered_data_clean['New catches'].sum():.0f} out of {len(engineered_data_clean)} ({engineered_data_clean['New catches'].mean()*100:.1f}%)")

# Feature availability check
print(f"\nFEATURE AVAILABILITY:")
key_features = ['Average Temperature', 'Average Humidity', 'Temp_Range', 'Season', 'Month', 'Day']
for feature in key_features:
    if feature in engineered_data_clean.columns:
        print(f"✓ {feature}: Available")
    else:
        print(f"✗ {feature}: Missing")

print(f"\n" + "=" * 60)
print("🎯 DATA PREPROCESSING COMPLETE!")
print("=" * 60)
print("\n📊 Both datasets are now clean and ready for modeling:")
print("   • Merged dataset: Ready for time-series analysis (SARIMAX)")
print("   • Engineered dataset: Ready for machine learning models")
print("\n🔍 Key insights from EDA:")
print("   • Pest activity varies significantly by location and time")
print("   • Weather conditions show correlations with pest captures")
print("   • Data spans multiple months with clear temporal patterns")
print("\n➡️  Ready to proceed to Notebook 2: Regression Modeling")


FINAL DATA VALIDATION & SUMMARY
DATA QUALITY CHECKS:
✓ Merged dataset: 245 records, 8 features
✓ Engineered dataset: 245 records, 17 features
✓ Date range: 2024-07-06 to 2024-08-23
✓ Number of locations: 5
✓ Locations: ['Cicalino 1', 'Cicalino 2', 'Imola 1', 'Imola 2', 'Imola 3']

TARGET VARIABLE SUMMARY:
• Total insect captures: 60
• Days with captures: 38 out of 245 (15.5%)
• Days with new catches: 26 out of 245 (10.6%)

FEATURE AVAILABILITY:
✓ Average Temperature: Available
✓ Average Humidity: Available
✓ Temp_Range: Available
✓ Season: Available
✓ Month: Available
✓ Day: Available

🎯 DATA PREPROCESSING COMPLETE!

📊 Both datasets are now clean and ready for modeling:
   • Merged dataset: Ready for time-series analysis (SARIMAX)
   • Engineered dataset: Ready for machine learning models

🔍 Key insights from EDA:
   • Pest activity varies significantly by location and time
   • Weather conditions show correlations with pest captures
   • Data spans multiple months with clear temporal

In [14]:
# Save cleaned datasets for modeling
print("\nSaving cleaned datasets...")

# Reset index for merged data to save Date as column
merged_data_final = merged_data_clean.reset_index()

# Save both datasets
merged_data_final.to_csv('cleaned_merged_data.csv', index=False)
engineered_data_clean.to_csv('cleaned_engineered_data.csv', index=False)

print("✓ Cleaned datasets saved:")
print("  • cleaned_merged_data.csv (for time-series modeling)")
print("  • cleaned_engineered_data.csv (for ML modeling)")
print("\n🎉 Notebook 1 Complete! Ready for modeling phase.")


Saving cleaned datasets...
✓ Cleaned datasets saved:
  • cleaned_merged_data.csv (for time-series modeling)
  • cleaned_engineered_data.csv (for ML modeling)

🎉 Notebook 1 Complete! Ready for modeling phase.

✓ Cleaned datasets saved:
  • cleaned_merged_data.csv (for time-series modeling)
  • cleaned_engineered_data.csv (for ML modeling)

🎉 Notebook 1 Complete! Ready for modeling phase.


---

**Summary:**  
- We loaded and cleaned our datasets to ensure quality.
- We explored the data visually to find patterns and relationships.
- These steps set the foundation for building accurate and useful predictive models in the next notebooks.

Thank you for following along!  
**Next:** We move to Notebook 2, where we build and evaluate regression models to predict pest activity.

---